- breakdown the model (add explainability)
- used with trained weights of the model 

In [7]:
import torch 
import torch.nn as nn 
import matplotlib.pyplot as plt 
from utils import * 
from data.data_load import data_process,load_data 
from data.tokenizer import init_tokenizer 
from model.rnn import LSTM 
import model.tokenizers as tokenizers 
from evaluator import extract_test_errors 


### Find test errors that model made 

In [2]:
checkpoint = load_checkpoint("checkpoints/8 Epochs 1 Layer LSTM Acc=0.74.pt")

In [3]:
for key in checkpoint.keys():
    print(key)

state_dict
config
pretrain_embed
test_acc


In [4]:
print(checkpoint["config"])
print(checkpoint["pretrain_embed"])

{'n_layers': 1, 'hidden_dim': 32, 'embed_dim': 32, 'output_dim': 3, 'epochs': 8, 'batch_size': 64, 'learning_rate': 0.001, 'dropOut': 0.5, 'weight_decay': 1e-05, 'model': 'LSTM', 'vocab_size': 28996}
bert-base-cased


load model and trained weights 

In [5]:
hyperparams = checkpoint["config"]
embed_weights= extract_bert_weight(checkpoint["pretrain_embed"])
model = LSTM(n_layers=hyperparams["n_layers"],
            embed_dim=hyperparams["embed_dim"],
            hidden_dim=hyperparams["hidden_dim"],
            output_dim=hyperparams["output_dim"],
            vocab_size=hyperparams["vocab_size"],
            dropOut=hyperparams["dropOut"],
            embedding_weights=embed_weights)

model.load_state_dict(checkpoint["state_dict"])

<All keys matched successfully>

load test data 

In [ ]:
df,_= load_data() 
tokenizer= init_tokenizer(model="bert-base-cased")
_, test_loader, _, _, test_size= data_process(df, 
                                    batch_size=hyperparams["batch_size"], 
                                    )

In [7]:
df = extract_test_errors(model, test_loader, tokenizer, fileName="8 Epochs 1 Layer LSTM Acc=0.74 error_analysis.csv")

Number of errors:252
----Error Analysis CSV Created----


### Chi-Square Feature pruning Examples 

In [1]:
import pandas as pd 
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
from data.data_load import load_data
from data.feature_pruning import * 

/opt/anaconda3/envs/newEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df,_ = load_data()

In [9]:
for i in range(3):
    print(df["news"][i])

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .
The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .


In [10]:
selected_features = chi_square_pruning(df,k=3000)

In [11]:
def prune_dataset(df, selected_features):
    '''  
    apply pruning on the dataset 
    @return pruned dataset for creating dataset and dataloader 
    '''
    header = df.columns.tolist()
    assert("news" in header)

    def prune(word_list):
        return " ".join([word if word.lower() in selected_features else "[UNK]" for word in word_list.split()])    
    

    #  given the input text, change words that didn't make into selected dictionary into [UNK] 
    # modify in place
    df["news"]= df["news"].apply(prune)
    return


In [12]:
prune_dataset(df, selected_features)

In [13]:
for i in range(3):
    print(df["news"][i])

According to Gran [UNK] the company has no plans to move all [UNK] to Russia [UNK] although [UNK] is where the company is [UNK] [UNK]
[UNK] plans to develop in [UNK] an area [UNK] no less than [UNK] square meters in order to [UNK] companies working in computer technologies and telecommunications [UNK] the statement said [UNK]
The [UNK] [UNK] industry company [UNK] has laid [UNK] [UNK] [UNK] employees from its Tallinn facility [UNK] [UNK] to earlier [UNK] the company contracted the [UNK] [UNK] its [UNK] [UNK] [UNK] the daily [UNK] reported [UNK]
